# Titanic Survival Prediction — Annotated Notebook
**1. Load Data**


In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Group train and test together so any transformation below is applied
# to both and avoiding inconsistencies between them
train_test_data = [train, test]

**2. Extract Title from Name**

Names contain a title (Mr, Mrs, Miss, etc.)

In [2]:
title_mapping = {"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3,
                 "Dr": 3, "Rev": 3, "Col": 3, "Major": 3,
                 "Mlle": 3, "Countess": 3, "Ms": 3, "Lady": 3,
                 "Jonkheer": 3, "Don": 3, "Dona": 3, "Mme": 3,
                 "Capt": 3, "Sir": 3}

# Extract the title word from each Name
for dataset in train_test_data:
    dataset['Title'] = dataset['Name'].str.extract(' ([A-Za-z]+)', expand=False)

# Convert extracted title text into numeric group
for dataset in train_test_data:
    dataset['Title'] = dataset['Title'].map(title_mapping)

# Any title not in our mapping defaults to group 3
for dataset in train_test_data:
    dataset['Title'] = dataset['Title'].fillna(3)

# Name itself is no longer needed
train.drop('Name', axis=1, inplace=True)
test.drop('Name', axis=1, inplace=True)

**3. Encode Sex**

In [3]:
sex_mapping = {"male": 0, "female": 1}
for dataset in train_test_data:
    dataset['Sex'] = dataset['Sex'].map(sex_mapping)

**4. Fill Missing Age — Grouped by Title**

Instead of filling all missing ages with one overall median, fill using the median age within each Title group like a "Master" and a "Mr" might have different age

In [4]:
train["Age"] = train["Age"].fillna(train.groupby("Title")["Age"].transform("median"))
test["Age"] = test["Age"].fillna(test.groupby("Title")["Age"].transform("median"))

**5. Fill Missing Embarked & Encode**

In [5]:
for dataset in train_test_data:
    dataset['Embarked'] = dataset['Embarked'].fillna('S')

embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}
for dataset in train_test_data:
    dataset['Embarked'] = dataset['Embarked'].map(embarked_mapping)

**6. Fill Missing Fare — Grouped by Passenger Class**

Fare is closely tied to class, so filling missing fares using the median within each Pclass is more accurate than an overall median

In [6]:
train["Fare"] = train["Fare"].fillna(train.groupby("Pclass")["Fare"].transform("median"))
test["Fare"] = test["Fare"].fillna(test.groupby("Pclass")["Fare"].transform("median"))

**7. Encode Cabin (First Letter Only)**

Cabin numbers themselves aren't meaningful, but the first character(deck letter) might tell about class or location on the ship so we extract only that and encode it numerically.

In [7]:
cabin_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5, "G": 6, "T": 7}
for dataset in train_test_data:
    dataset['Cabin'] = dataset['Cabin'].str[:1]
    dataset['Cabin'] = dataset['Cabin'].map(cabin_mapping)

# Most cabin values are missing (around 77%)
# Fill using median deck with each Pclass since cabin is related to class
train["Cabin"] = train["Cabin"].fillna(train.groupby("Pclass")["Cabin"].transform("median"))
test["Cabin"] = test["Cabin"].fillna(test.groupby("Pclass")["Cabin"].transform("median"))

**8. Feature Engineering: Family Size & Alone Status**

In [8]:
# Combine SibSp (siblings/spouses) and Parch (parents/children) into one FamilySize
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

# Flag passengers traveling completely alone as survival patterns may
# differ meaningfully for solo travelers
for dataset in train_test_data:
    dataset['IsAlone'] = (dataset['FamilySize'] == 1).astype(int)

**9. Feature Engineering: HasCabin Flag**

Rather than only relying on the filled-in Cabin deck value (which is mostly guessed for 77% of passengers), also added a simple flag for whether a cabin was recorded at all

In [9]:
# Re-load raw data just to check original missingness before we filled it in
train_raw = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_raw = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

train['HasCabin'] = train_raw['Cabin'].notna().astype(int)
test['HasCabin'] = test_raw['Cabin'].notna().astype(int)

**10. Drop Unused Columns & Prepare Final Feature Set**

In [10]:
# Ticket (mostly unique and low predictive value on its own),
# SibSp and Parch are no longer needed
features_drop = ['Ticket', 'SibSp', 'Parch']
train = train.drop(features_drop, axis=1)
test = test.drop(features_drop, axis=1)

# Keep PassengerId aside for the final submission file and not using it
passenger_ids = test['PassengerId']
train = train.drop(['PassengerId'], axis=1)
X_test = test.drop(['PassengerId'], axis=1)

X_train = train.drop('Survived', axis=1)
y_train = train['Survived']

# Ensure test set columns are in the exact same order as train,
# so the model sees consistent feature positions across both
X_test = X_test[X_train.columns]

**11. Compare Multiple Models with Cross-Validation**

Rather than picking one model upfront, we compare several using 5 fold cross-validation

In [11]:
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier

models = {
    'LogisticRegression': LogisticRegression(C=0.1, max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0),
    'XGBoost': XGBClassifier(random_state=0, eval_metric='logloss', max_depth=3, learning_rate=0.05, n_estimators=200),
    'GradientBoosting': GradientBoostingClassifier(random_state=0),
}

print("Individual model scores (5-fold CV):")
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy')
    print(f"  {name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Individual model scores (5-fold CV):
  LogisticRegression: 0.8171 (+/- 0.0161)
  RandomForest: 0.8126 (+/- 0.0201)
  XGBoost: 0.8272 (+/- 0.0270)
  GradientBoosting: 0.8238 (+/- 0.0298)


**12. Ensemble: Combine Models via Soft Voting**

Instead of relying on a single best model, a soft-voting ensemble averages the predicted probabilities from multiple models which is often more robust than any one model alone

In [12]:
ensemble = VotingClassifier(estimators=[
    ('lr', models['LogisticRegression']),
    ('rf', models['RandomForest']),
    ('xgb', models['XGBoost']),
], voting='soft')

ensemble_scores = cross_val_score(ensemble, X_train, y_train, cv=5, scoring='accuracy')
print(f"\nEnsemble (soft voting): {ensemble_scores.mean():.4f} (+/- {ensemble_scores.std():.4f})")


Ensemble (soft voting): 0.8294 (+/- 0.0125)


**13. Pick the Best Performing Model/Ensemble**

In [13]:
all_results = {name: cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy').mean()
               for name, m in models.items()}
all_results['Ensemble'] = ensemble_scores.mean()

best_name = max(all_results, key=all_results.get)
print(f"\nBest overall: {best_name} ({all_results[best_name]:.4f})")


Best overall: Ensemble (0.8294)


**14. Train Final Model & Generate Submission**

In [14]:
# Re-fit the winning model/ensemble on the FULL training data
# (cross-validation only used it in folds; now we use all of it for the final model)
best_model = ensemble if best_name == 'Ensemble' else models[best_name]
best_model.fit(X_train, y_train)
predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv created!")

submission.csv created!
